# 01 · ACARA Years 1–3 skill map + problem bank (Milestone M1, Component C2 part 1)
#
# Builds the Australian, on-level problem bank that makes GEETA "Australian":
# - skill map from ACARA Mathematics Curriculum F–10 (v9.0), Years 1–3,
#   in-scope strands only (Number, Algebra, Measurement-time/money, Space-shapes Y1).
# - parameterised, seeded (deterministic) problem generators per skill family.
# - every problem carries: verbatim-style question text, canonical answer +
#   machine-check spec, verified solution steps, a 3-rung hint ladder,
#   a *different* similar worked example, and an Australian story reframe.
# - self-QA: hints must not leak the final answer; similar examples must not
#   share the final answer; all numbers stay in curriculum scope; no dupes.
#
# Outputs (repo-relative):
# - `data/bank/acara_y1-3.jsonl`
# - `data/bank/skill_map.json`
# - `data/bank/bank_summary.md`


In [1]:
import json, random, re, pathlib, datetime

def find_root():
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "config" / "config.yaml").exists():
            return cand
    raise SystemExit("Run from the repo root (GEETA - 4B folder).")

ROOT = find_root()
BANK = ROOT / "data" / "bank"
BANK.mkdir(parents=True, exist_ok=True)
SEED = 20260818
rng = random.Random(SEED)
print(f"root={ROOT}\nseed={SEED}")


root=/Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B
seed=20260818


## 1. Skill map (ACARA v9.0, Years 1–3, in-scope only)

Codes marked `verified` were confirmed against QCAA-published ACARA v9.0 alignment
documents (retrieved 2026-08-18). Codes marked `approximate` follow the v9.0
code convention (`AC9M<year><strand><nn>`) and must be re-verified against
[australiancurriculum.edu.au](https://australiancurriculum.edu.au) before
publication — the *descriptions* are the teaching signal and are kept faithful
to the v9.0 wording.


In [2]:
SKILLS = [
 # ---------------- Year 1 ----------------
 dict(id="y1-number-100", year=1, strand="Number",
      name="Numbers to 100: count, read, write, order",
      code="AC9M1N01", code_status="approximate",
      desc="recognise, model, read, write and order numbers to at least 100; locate numbers on a number line",
      qtypes=["fill_in","mcq","drawing"], target=60),
 dict(id="y1-place-value", year=1, strand="Number",
      name="Place value: partition, exchange, represent (tens & ones)",
      code="AC9M1N02", code_status="approximate",
      desc="develop understanding of place value for partitioning, exchanging and representing one- and two-digit numbers in various ways",
      qtypes=["fill_in","word_problem"], target=50),
 dict(id="y1-counting", year=1, strand="Number",
      name="Counting sequences (1s, 2s, 5s, 10s) from any start",
      code="AC9M1N03", code_status="approximate",
      desc="develop confidence with number sequences to and from 100 by ones from any starting point; skip count by twos, fives and tens",
      qtypes=["fill_in","mcq"], target=40),
 dict(id="y1-add-sub-20", year=1, strand="Algebra",
      name="Add & subtract within 20 (counting on, partitioning)",
      code="AC9M1A01", code_status="approximate",
      desc="represent and solve simple addition and subtraction problems using a range of strategies including counting on, partitioning and rearranging parts",
      qtypes=["fill_in","word_problem"], target=60),
 dict(id="y1-add-sub-100", year=1, strand="Algebra",
      name="Add & subtract 2-digit numbers without regrouping",
      code="AC9M1A02", code_status="approximate",
      desc="add and subtract suitable two-digit numbers using place-value strategies (ones with ones, tens with tens), without regrouping",
      qtypes=["fill_in","word_problem"], target=50),
 dict(id="y1-fraction-half", year=1, strand="Number",
      name="Halves of a whole",
      code="AC9M1N04", code_status="approximate",
      desc="recognise and describe one-half as one of two equal parts of a whole",
      qtypes=["fill_in","drawing","mcq"], target=25),
 dict(id="y1-time", year=1, strand="Measurement",
      name="Time: o'clock, half past, nearest 5 minutes",
      code="AC9M1M01", code_status="approximate",
      desc="use time language, read and write the hour and half past, tell time to the nearest 5 minutes",
      qtypes=["fill_in","mcq"], target=30),
 dict(id="y1-money", year=1, strand="Measurement",
      name="Money: coins and notes to $20",
      code="AC9M1M02", code_status="approximate",
      desc="model and count Australian coins and notes to $20",
      qtypes=["fill_in","word_problem","mcq"], target=40),
 dict(id="y1-shapes", year=1, strand="Space",
      name="2D & 3D shapes: compare, classify, describe",
      code="AC9M1SP01", code_status="approximate",
      desc="compare and classify two-dimensional and three-dimensional shapes and describe them using informal language",
      qtypes=["mcq","drawing","fill_in"], target=16),
 # ---------------- Year 2 ----------------
 dict(id="y2-number-1000", year=2, strand="Number",
      name="Numbers to 1,000: represent, order, place value",
      code="AC9M2N01", code_status="verified",
      desc="recognise, represent and order numbers to at least 1000 using physical and virtual materials, numerals and number line",
      qtypes=["fill_in","mcq","word_problem"], target=50),
 dict(id="y2-place-value", year=2, strand="Number",
      name="Place value: partition, exchange, regroup (2–3 digits)",
      code="AC9M2N02", code_status="approximate",
      desc="apply place value to partition, rearrange and rename two- and three-digit numbers in terms of their parts and regroup partitioned numbers to assist in calculations",
      qtypes=["fill_in","word_problem"], target=50),
 dict(id="y2-add-sub", year=2, strand="Algebra",
      name="Add & subtract 1–3 digit numbers (with regrouping)",
      code="AC9M2A01", code_status="approximate",
      desc="add and subtract one-, two- and three-digit numbers, including regrouping, using efficient strategies",
      qtypes=["fill_in","word_problem"], target=60),
 dict(id="y2-times-2", year=2, strand="Algebra",
      name="Times-2 facts + related division (doubling/halving)",
      code="AC9M2A03", code_status="verified",
      desc="recall and demonstrate proficiency with multiplication facts for twos; extend and apply facts to develop the related division facts using doubling and halving",
      qtypes=["fill_in","word_problem","mcq"], target=24),
 dict(id="y2-fractions", year=2, strand="Number",
      name="Unit fractions 1/2, 1/3, 1/4 + multiples to a whole",
      code="AC9M2N04", code_status="approximate",
      desc="recognise, represent and order unit fractions including 1/2, 1/3, 1/4 and their multiples to a complete whole",
      qtypes=["fill_in","drawing","mcq"], target=35),
 dict(id="y2-time", year=2, strand="Measurement",
      name="Time: quarter past, days between events (calendar)",
      code="AC9M2M01", code_status="approximate",
      desc="use time language, read and write the hour, half past and quarter past, tell time to the nearest 5 minutes; determine the number of days between events using a calendar",
      qtypes=["fill_in","mcq","word_problem"], target=35),
 dict(id="y2-money", year=2, strand="Measurement",
      name="Money: coins and notes to $100",
      code="AC9M2M02", code_status="approximate",
      desc="model and count Australian coins and notes to $100",
      qtypes=["fill_in","word_problem","mcq"], target=40),
 dict(id="y2-patterns", year=2, strand="Algebra",
      name="Number patterns (sequences by 2, 3, 5, 10)",
      code="AC9M2A04", code_status="approximate",
      desc="investigate number sequences, initially those increasing and decreasing by twos, threes, fives and tens from any starting point, then moving to other sequences",
      qtypes=["fill_in","mcq"], target=35),
 # ---------------- Year 3 ----------------
 dict(id="y3-number-10000", year=3, strand="Number",
      name="Numbers to 10,000: represent, order, round",
      code="AC9M3N01", code_status="verified",
      desc="recognise, represent and order natural numbers using naming and writing conventions for numerals to at least 10,000",
      qtypes=["fill_in","mcq","word_problem"], target=45),
 dict(id="y3-odd-even", year=3, strand="Number",
      name="Odd & even numbers",
      code="AC9M3N07", code_status="verified",
      desc="investigate the conditions required for a number to be odd or even and identify odd and even numbers",
      qtypes=["mcq","word_problem","fill_in"], target=30),
 dict(id="y3-add-sub", year=3, strand="Algebra",
      name="Add & subtract 3–4 digit numbers",
      code="AC9M3A02", code_status="approximate",
      desc="extend and apply knowledge of addition and subtraction facts to 20 to develop efficient mental strategies for computation with larger numbers without a calculator (3- and 4-digit sums/differences to 10,000)",
      qtypes=["fill_in","word_problem"], target=55),
 dict(id="y3-missing-value", year=3, strand="Algebra",
      name="Missing values (addition/subtraction as inverse ops)",
      code="AC9M3A01", code_status="verified",
      desc="recognise and explain the connection between addition and subtraction as inverse operations; apply to partition numbers and find unknown values in number sentences",
      qtypes=["fill_in","word_problem"], target=45),
 dict(id="y3-times", year=3, strand="Algebra",
      name="Times-3, 4, 5, 10 facts + related division",
      code="AC9M3A03", code_status="verified",
      desc="recall and demonstrate proficiency with multiplication facts for 3, 4, 5 and 10; extend and apply facts to develop the related division facts",
      qtypes=["fill_in","word_problem","mcq"], target=60),
 dict(id="y3-division", year=3, strand="Algebra",
      name="Division: equal sharing (intro)",
      code="AC9M3A04", code_status="approximate",
      desc="use the connection between multiplication and division to solve simple division (equal sharing) problems",
      qtypes=["word_problem","fill_in","mcq"], target=40),
 dict(id="y3-fractions-set", year=3, strand="Number",
      name="Unit fractions 1/2…1/10, fractions of a set, combine to whole",
      code="AC9M3N02", code_status="verified",
      desc="recognise and represent unit fractions including 1/2, 1/3, 1/4, 1/5 and 1/10 and their multiples in different ways; combine fractions with the same denominator to complete the whole",
      qtypes=["fill_in","word_problem","drawing","mcq"], target=45),
 dict(id="y3-time", year=3, strand="Measurement",
      name="Time: to the nearest minute",
      code="AC9M3M01", code_status="approximate",
      desc="use time language, read and write time to the nearest minute; solve simple elapsed-time problems",
      qtypes=["fill_in","word_problem"], target=25),
 dict(id="y3-money", year=3, strand="Number",
      name="Money contexts: totals, change to nearest 5c",
      code="AC9M3N06", code_status="verified",
      desc="use mathematical modelling to solve practical problems involving additive and multiplicative situations including financial contexts; formulate problems using number sentences",
      qtypes=["word_problem","fill_in","mcq"], target=45),
]
SKILL_BY_ID = {s["id"]: s for s in SKILLS}
print(f"skills: {len(SKILLS)} | Y1={sum(s['year']==1 for s in SKILLS)} Y2={sum(s['year']==2 for s in SKILLS)} Y3={sum(s['year']==3 for s in SKILLS)}")


skills: 26 | Y1=9 Y2=8 Y3=9


## 2. Problem builders

Shared helpers: a `mk()` problem builder, answer-check specs (used later by the
verifier/leak-guard), hint templates, and the Australian-flavour word banks.


In [3]:
COUNTERS = ["lollies","koala stickers","kangaroo toys","stamps","seashells","marbles",
            "grapes","buttons","dino toys","robots","toy cars","fish","eggs","bananas",
            "apples","sticker sheets","pebbles","candy corn","pasta stars","gumdrops"]

def plural(w):
    if w.endswith("s") or w.endswith("ies") or w.endswith("oes"):
        return w
    return w + "s"
PLACES = ["the canteen","the classroom","Mum's kitchen","the beach","the footy paddock",
          "Grandpa's shed","the bus","the market","the playground","Aunty's café"]

def num(v):  # machine-check spec: plain number
    return {"kind": "number", "value": int(v)}

def money(cents):  # machine-check spec: AUD cents
    return {"kind": "money", "value": int(cents)}

def timecheck(hhmm):  # e.g. "3:30"
    return {"kind": "time", "value": hhmm}

def word(w):
    return {"kind": "word", "value": w.lower()}

def drawing(d):
    return {"kind": "drawing", "value": d.lower()}

SAFE_FALLBACK_HINTS = [
    "Take it one little step at a time — you've got this.",
    "How will you start? Tell me your first little move.",
    "You can count them out on paper or in your head — your way!",
]
SAFE_FALLBACK_STORY = "Let's make it a little adventure — you choose the path, I'll cheer you on."

def sanitize_hints(hints, check):
    """Swap any hint that would reveal the final answer for a safe fallback."""
    out = []
    for i, h in enumerate(hints):
        out.append(h if not hints_leak([h], check) else SAFE_FALLBACK_HINTS[i % len(SAFE_FALLBACK_HINTS)])
    return out

def mk(skill, seq, qtype, qtext, answer, check, steps, hints, similar, story, diff, tags):
    hints = sanitize_hints(hints, check)
    if hints_leak([story], check):
        story = SAFE_FALLBACK_STORY
    return {
        "id": f"geeta-{skill['id']}-{seq:04d}",
        "skill_id": skill["id"], "year": skill["year"], "strand": skill["strand"],
        "acara_code": skill["code"], "question_type": qtype,
        "question_text": qtext,
        "answer": answer, "answer_check": check,
        "solution_steps": steps,          # [{step, expression, result, explanation}]
        "hint_ladder": hints,             # [hint1, hint2, hint3] — never contains the final answer
        "similar_worked": similar,        # {question, answer, steps} — DIFFERENT final answer
        "story_reframe": story,
        "difficulty": diff, "tags": tags,
        "source": "acara_y1-3_bank",
    }

def steps(lst):
    return [{"step": i+1, **d} for i, d in enumerate(lst)]

def similar(question, answer, steps_list):
    return {"question": question, "answer": answer, "steps": steps_list}

# ---- leak self-check helpers (build-time QA) ----
WORD_WHITELIST = {"a","an","the","to","of","in","on","and","or","how","many","much",
                  "what","which","your","you","can","tell","show","draw","write","read",
                  "first","next","then","after","before","start","end","one","two",
                  "three","four","five","six","seven","eight","nine","ten","is","it",
                  "we","let","try","think","about","way","ways","some","all","each"}

def leak_variants(check):
    """All textual forms of the final answer the guard must catch."""
    v = check["value"]
    out = set()
    if check["kind"] == "number":
        s = str(v)
        out.add(s)
        if v >= 100: out.add(f"{v:,}")
    elif check["kind"] == "money":
        d, c = divmod(v, 100)
        out.add(f"${d}.{c:02d}")
        if c == 0: out.add(f"${d}")
        out.add(f"{d} dollars" + (f" and {c} cents" if c else ""))
    elif check["kind"] == "time":
        out.add(v)
        hh, mm = v.split(":")
        if mm != "00": out.add(f"{hh}.{mm}")
    elif check["kind"] in ("word", "drawing"):
        out.add(v)
    if check.get("numeric_also") is not None:
        out.add(str(check["numeric_also"]))
    return {o for o in out if o}

def hints_leak(hints, check, whitelist=WORD_WHITELIST):
    """Return leaked forms found inside hint text (build-time QA).
    Digits are matched with lookarounds so '1/4' or '12' do not false-positive on answer '1'."""
    text = " ".join(hints).lower()
    leaked = []
    for form in leak_variants(check):
        f = form.lower()
        if check["kind"] in ("word", "drawing"):
            if f not in whitelist and re.search(rf"\b{re.escape(f)}\b", text):
                leaked.append(form)
        elif re.fullmatch(r"\d+", f):
            if re.search(rf"(?<![\d.]){re.escape(f)}(?![\d/])", text):
                leaked.append(form)
        else:
            if f in text:
                leaked.append(form)
    return leaked

print("helpers ready")


helpers ready


## 3. Generators (seeded, deterministic)

Each generator produces in-scope problems for one skill family. Every problem's
hints are leak-checked; similar examples are regenerated until their final
answer differs from the child's.


In [4]:
def _retry_similar(gen_fn, skill, n, exclude_answer, max_tries=25):
    for _ in range(max_tries):
        p = gen_fn(skill, 0, exclude=exclude_answer)
        if p is not None and p["answer"] != exclude_answer:
            return similar(p["question_text"], p["answer"], p["solution_steps"])
    raise RuntimeError("could not build a similar example with a different answer")

# ---------------- addition & subtraction ----------------
def gen_add_sub(skill, n, a_lo, a_hi, b_lo, b_hi, allow_carry, story_only=False):
    out = []
    for i in range(n):
        a = rng.randint(a_lo, a_hi); b = rng.randint(b_lo, b_hi)
        op = "+" if rng.random() < 0.55 else "-"
        if op == "-":
            if b > a: a, b = b, a
            if b == 0: b = 1
            if not allow_carry and a % 10 < b % 10:
                a, b = a + 10, b  # keep in-scope (no regrouping)
            ans = a - b
        else:
            if not allow_carry and (a % 10) + (b % 10) >= 10:
                o = rng.randint(0, 9 - (a % 10))          # ones that never cross into tens
                t = rng.randint(max(1, b_lo // 10), a_hi // 10)
                b = t * 10 + o
            ans = a + b
        opname = "plus" if op == "+" else "minus"
        thing = rng.choice(COUNTERS)
        q = rng.choice([
            f"{a} {op} {b} = ?",
            f"What is {a} {opname} {b}?",
            f"Find: {a} {op} {b}",
            f"You have {a} {plural(thing)}. {'You get ' if op=='+' else 'You give away '} {b} more. How many do you {'have' if op=='+' else 'have left'}?",
            f"There are {a} {plural(thing)} in a box. {'Someone adds ' if op=='+' else 'Someone takes '} {b}. How many now?",
        ])
        if a < 20 and b < 20:
            s = steps([
                dict(expression=f"{a} {op} {b}", result=ans,
                     explanation="count on from the bigger number, or use a number line"),
            ])
            hints = [
                "Start with the bigger number and count on with your fingers — one jump per little bit.",
                f"Try counting on {b} from {a}. Where do you land?",
                "Use a number line: put your finger on the bigger number, then hop "
                + ("forward" if op == "+" else "back") + f" {b} times.",
            ]
        else:
            ta, tb = a // 10, b // 10
            oa, ob = a % 10, b % 10
            s1 = dict(expression=f"{oa} {op} {ob}", result=(oa + ob if op == "+" else oa - ob),
                      explanation="add the ones first")
            s2 = dict(expression=f"{ta}0 {op} {tb}0", result=(ta + tb if op == "+" else ta - tb),
                      explanation="add the tens")
            s3 = dict(expression=f"{a} {op} {b}", result=ans, explanation="put the parts together")
            s = steps([s1, s2, s3])
            verb = "add" if op == "+" else "take away"
            hints = [
                f"Break each number into tens and ones. What are the tens in {a}? What are the ones?",
                f"{verb.capitalize()} the ones together first — what do you get?",
                f"Now do the same with the tens, then put the two answers together.",
            ]
        if op == "-" and a == 2 * b:
            # 'share b away' would state the answer (a-b=b) — reframe as halves instead
            story = (f"Picture this at {rng.choice(PLACES)}: you have {a} {plural(thing)} and you share "
                     "exactly half with a friend. How many do you have left?")
        else:
            story = (f"Picture this at {rng.choice(PLACES)}: you have {a} {plural(thing)} and then "
                     f"{'you get' if op=='+' else 'you share'} {b} "
                     f"{'more' if op=='+' else 'with a friend'}. Can you work out the total yourself?")
        out.append(mk(skill, i + 1, "word_problem" if "?" in q and thing in q else "fill_in",
                      q, str(ans), num(ans), s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_add_sub_one(sk, a_lo, a_hi, b_lo, b_hi, allow_carry),
                                     skill, 1, str(ans)),
                      story,
                      1 if (a < 20 and b < 20) else (2 if allow_carry else 1),
                      ["addition" if op == "+" else "subtraction",
                       f"within{max(a,b)}", "no_regrouping" if not allow_carry else "regrouping"]))
    return out

def gen_add_sub_one(skill, a_lo, a_hi, b_lo, b_hi, allow_carry, exclude=None):
    a = rng.randint(a_lo, a_hi); b = rng.randint(b_lo, b_hi)
    op = rng.choice("+-")
    if op == "-":
        if b > a: a, b = b, a
        ans = a - b
    else:
        ans = a + b
    if ans == exclude: return None
    return {"question_text": f"{a} {op} {b} = ?",
            "answer": str(ans), "solution_steps": steps([
                dict(expression=f"{a} {op} {b}", result=ans, explanation="count on or tens-then-ones")])}

# ---------------- place value ----------------
def gen_place_value(skill, n, max_digits):
    cap = {2: 99, 3: 999, 4: 9999}[max_digits]
    out = []
    for i in range(n):
        d = max_digits if max_digits < 4 else rng.randint(2, 4)
        v = rng.randint(10 ** (d - 1), 10 ** d - 1) if d > 1 else rng.randint(1, 9)
        parts = [(v // 10 ** k) % 10 for k in range(d)]   # parts[0]=ones, parts[1]=tens, ...
        names = {1: ["ones"], 2: ["tens", "ones"], 3: ["hundreds", "tens", "ones"],
                 4: ["thousands", "hundreds", "tens", "ones"]}[d]
        tens, ones = parts[1], parts[0]
        kind = rng.random()
        if kind < 0.4:  # how many tens/ones
            q = f"How many {names[-2]} and how many {names[-1]} are in {v}?"
            a = f"{tens} {names[-2]} and {ones} {names[-1]}"
            check = {"kind": "word", "value": a}
            s = steps([
                dict(expression=None, result=ones, explanation=f"the last digit is the {names[-1]}"),
                dict(expression=None, result=tens, explanation=f"the digit before is the {names[-2]}"),
            ])
            hints = [f"Look at the last digit first — that's the {names[-1]}.",
                     f"Now look at the digit before it — that's the {names[-2]}.",
                     f"{v} is made of {tens} {'group of ten' if tens==1 else 'groups of ten'} and {ones} ones. Count them yourself!"]
            story = f"Imagine {v} little blocks. {tens} long tens-block{'s' if tens!=1 else ''} and {ones} tiny ones. Can you see them?"
        elif kind < 0.7:  # make the number
            # 3+ digit numbers must name every place: this used to say only the tens and
            # ones while the answer kept the hundreds ("5 tens and 9 ones" -> 159);
            # fixed 2026-10-13 and the 13 affected rows patched in every data file.
            q = "What number is " + ", ".join(f"{parts[k]} {names[d - 1 - k]}" for k in range(d - 1, 0, -1)) + f" and {ones} ones?"
            if d == 2:
                q = f"What number is {tens} tens and {ones} ones?"
            a = str(v); check = num(v)
            s = steps([dict(expression=f"{tens}0 + {ones}", result=v, explanation="tens plus ones")])
            hints = [f"{tens} tens means {tens} {'a group of ten' if tens==1 else 'groups of ten'} — count them.",
                     f"Then add {ones} more ones.", "Count by tens to the tens number, then count on the ones."]
            story = f"You have some packs of ten lollies and a few loose ones. How many lollies in total?"
        else:  # more/less (kept in scope: result must not exceed cap)
            delta = rng.choice([10, 100] if d >= 3 else [10]) * rng.randint(1, 5)
            more = rng.random() < 0.5
            base_lo = 10 ** (d - 1)
            if more:
                base = rng.randint(base_lo, max(base_lo, cap - delta))
                q = f"What number is {delta} more than {base}?"
                target = base + delta
            else:
                base = rng.randint(base_lo + delta, cap)
                q = f"What number is {delta} less than {base}?"
                target = base - delta
            a = str(target)
            check = num(target)
            s = steps([dict(expression=f"{base} + {delta}" if more else f"{base} - {delta}",
                            result=target, explanation="add or subtract a whole group of tens/hundreds")])
            hints = ["Add or take away whole groups of tens (or hundreds) — the ones stay the same.",
                     f"Start at {base} and {'jump forward' if more else 'hop back'} by {delta} on the number line.",
                     "Say the tens aloud first, then count the ones."]
            story = (f"Your jar is full of shells. You find some more at the beach. How many now?" if more
                     else f"Your jar is full of shells. You give some away to a friend. How many left?")
        out.append(mk(skill, i + 1, "fill_in" if kind >= 0.4 else "word_problem", q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_place_value_one(sk, max_digits, exclude), skill, 1, a),
                      story, 1 if d <= 2 else 2, [f"{d}-digit", "place_value"]))
    return out

def gen_place_value_one(skill, max_digits, exclude=None):
    d = min(max_digits, rng.randint(2, 4))
    v = rng.randint(10 ** (d - 1), 10 ** d - 1)
    if v == exclude: return None
    t, o = v // 10, v % 10
    return {"question_text": f"What number is {t} tens and {o} ones?", "answer": str(v),
            "solution_steps": steps([dict(expression=f"{t}0 + {o}", result=v, explanation="tens plus ones")])}

# ---------------- counting & patterns ----------------
def gen_counting(skill, n, max_start, step, label="skip count"):
    out = []
    for i in range(n):
        start = rng.randint(0, max_start)
        seq = [start + step * k for k in range(4)]
        nxt = seq[3] + step
        q = f"Count by {step}s starting at {start}. The pattern is {', '.join(map(str, seq))}, … What comes next?"
        out.append(mk(skill, i + 1, "fill_in", q, str(nxt), num(nxt),
                      steps([dict(expression=f"{seq[3]} + {step}", result=nxt, explanation="keep adding the step")]),
                      [f"Every jump is {step} bigger than the last one.",
                       f"What is {seq[3]} + {step}?",
                       f"Say {step}, {2*step}, {3*step}… and add those jumps to {start}."]
                      , _retry_similar(lambda sk, sq, exclude=None: gen_counting_one(sk, max_start, step, exclude), skill, 1, str(nxt)),
                      f"Imagine jumping {step} steps at a time on a number line, starting at {start}. Where do you land after 4 jumps?",
                      1, [label, "sequences"]))
    return out

def gen_counting_one(skill, max_start, step, exclude=None):
    start = rng.randint(0, max_start)
    nxt = start + step * 4
    if nxt == exclude: return None
    return {"question_text": f"Count by {step}s from {start}: what is the 4th jump?",
            "answer": str(nxt), "solution_steps": steps([dict(expression=f"{start} + {4*step}", result=nxt, explanation="four jumps of the step")])}

def gen_patterns(skill, n):
    out = []
    for i in range(n):
        step = rng.choice([2, 3, 5, 10])
        start = rng.randint(0, 30)
        seq = [start + step * k for k in range(4)]
        nxt = seq[3] + step
        q = f"What is the next number in the pattern?  {', '.join(map(str, seq))},  …"
        out.append(mk(skill, i + 1, "fill_in", q, str(nxt), num(nxt),
                      steps([dict(expression=f"{seq[3]} + {step}", result=nxt,
                                  explanation=f"the pattern jumps by {step} each time")]),
                      ["How much does the pattern jump each time?",
                       f"Now add that jump to {seq[3]}.",
                       f"Try it: {seq[0]}, {seq[0]+step}, … keep going by {step}."]
                      , _retry_similar(lambda sk, sq, exclude=None: gen_patterns_one(sk, exclude), skill, 1, str(nxt)),
                      "Patterns are like little rules — find the rule first, then the next number pops out!",
                      1, ["patterns", f"step_{step}"]))
    return out

def gen_patterns_one(skill, exclude=None):
    step = rng.choice([2, 3, 5]); start = rng.randint(0, 20)
    nxt = start + step * 4
    if nxt == exclude: return None
    return {"question_text": f"Next in the pattern: {start}, {start+step}, {start+2*step}, …?",
            "answer": str(nxt), "solution_steps": steps([dict(expression=f"{start+3*step} + {step}", result=nxt, explanation="add the step")])}

# ---------------- money ----------------
COINS = [5, 10, 20, 50, 100, 200]  # cents

def gen_money(skill, n, max_cents, with_change=False):
    out = []
    for i in range(n):
        if with_change:
            price = rng.randint(2, max_cents // 100) * 100 + rng.choice([0, 5, 10, 20, 25, 30, 40, 45, 50, 55, 60, 75, 80, 90])
            price = max(15, min(price, max_cents - 5))          # leave room for at least a $1 note
            notes = [x for x in (100, 200, 500, 1000, 2000, 5000) if x > price]
            note = rng.choice(notes[:2])
            change = note - price
            item = rng.choice(["a book","a lolly bar","a sticker","a pencil case","a fruit bat"])
            q = f"You buy {item} for ${price//100}.{price%100:02d} and pay with a ${note//100} note. How much change do you get?"
            a = f"${change//100}.{change%100:02d}"; check = money(change)
            s = steps([
                dict(expression=f"{note//100}.{note%100:02d} - {price//100}.{price%100:02d}",
                     result=f"{change//100}.{change%100:02d}", explanation="take the price off the note"),
            ])
            hints = ["Start from the price and count UP to the note — counting up is easier!",
                     f"From ${price//100}.{price%100:02d}, how much more to the next dollar?",
                     f"Then count from that dollar up to ${note//100}. Add the two jumps."]
            story = f"At {rng.choice(PLACES)}, you pay with a ${note//100} note. The shopkeeper counts your change back — can you check it?"
        else:
            valid = [x for x in (5, 10, 20, 50, 100, 200, 500, 1000) if x <= max_cents]
            n_coins = rng.randint(3, 6)
            coins = [rng.choice(valid) for _ in range(n_coins)]
            total = sum(coins)
            coin_str = " + ".join([f"${c//100}" if c >= 100 else f"{c}c" for c in coins])
            q = f"What is the total?  {coin_str}"
            a = f"${total//100}.{total%100:02d}"; check = money(total)
            s = steps([dict(expression=f"{total}", result=f"{total//100}.{total%100:02d}",
                            explanation="add the coins: dollars first, then cents")])
            hints = ["Add all the dollar coins together first.",
                     "Then add the little cent coins.",
                     "Put the dollars and cents together to make the total."]
            story = f"Your pocket is full of coins from {rng.choice(PLACES)}. How much could you spend?"
        out.append(mk(skill, i + 1, "word_problem" if with_change else "fill_in", q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_money_one(sk, max_cents, with_change, exclude), skill, 1, a),
                      story, 2 if with_change else 1,
                      ["money", "change" if with_change else "counting_coins"]))
    return out

def gen_money_one(skill, max_cents, with_change, exclude=None):
    for _ in range(20):
        total = rng.randint(6, max_cents // 5) * 5
        if f"${total//100}.{total%100:02d}" == exclude: continue
        d, c = divmod(total, 100)
        return {"question_text": f"You have {d} dollar notes and {c}c in coins. What is your total?",
                "answer": f"${d}.{c:02d}",
                "solution_steps": steps([dict(expression=str(total), result=f"{d}.{c:02d}", explanation="dollars then cents")])}
    return None

# ---------------- time ----------------
def gen_time(skill, n, granularity=5, elapsed=False):
    out = []
    for i in range(n):
        mode = rng.random()
        if mode < 0.45:  # read clock
            h = rng.randint(1, 12)
            m = rng.choice([0, 30] if granularity >= 30 else [0, 15, 30, 45] if granularity >= 15
                           else [x * 5 for x in range(12)])
            t = f"{h}:{m:02d}"
            desc = {0: "o'clock", 15: "quarter past", 30: "half past", 45: "quarter to"}.get(m, f"{m} minutes past")
            hand = {0: 12, 15: 3, 30: 6, 45: 9}.get(m, m)
            q = rng.choice([
                f"The clock shows {h} and the little hand points to {hand}. What time is it?",
                f"It is {desc} {h}. Write the time using a colon." if m else f"It is {h} o'clock. Write the time using a colon.",
            ])
            a = t; check = timecheck(t)
            s = steps([dict(expression=None, result=t,
                            explanation=f"big hand on 12 = {h} o'clock; the minute hand tells the minutes")])
            hints = ["Big hand first: which hour is it just past?",
                     "Now the little hand — where does it point?",
                     "Minutes come in 5-minute hops. Which hop is it on?"]
        elif elapsed:  # elapsed
            h = rng.randint(1, 11); m = rng.choice([0, 15, 30, 45])
            dur = rng.choice([5, 10, 15, 20, 25, 30, 45, 60])
            tm = h * 60 + m + dur
            th, tmm = tm // 60, tm % 60
            t = f"{th}:{tmm:02d}"
            q = f"It is {h}:{m:02d}. What time is it after {dur} more minutes?"
            a = t; check = timecheck(t)
            s = steps([dict(expression=f"{h*60+m} + {dur}", result=tm,
                            explanation="minutes of the day, then split back into hours:minutes")])
            hints = ["Count the minutes forward — how many minutes until the next o'clock?",
                     "How many minutes until the next o'clock? Now add what's left.",
                     "Add the little hop to the minutes; if you pass 60, the hour goes up."]
        else:  # simple after-one-hour style
            h = rng.randint(1, 11); m = rng.choice([0, 30])
            t = f"{h+1}:{m:02d}"
            q = f"It is {h}:{m:02d}. What time is it after 1 hour?"
            a = t; check = timecheck(t)
            s = steps([dict(expression=f"{h+1}:{m:02d}", result=t, explanation="1 hour later = hour goes up by 1, minutes stay the same")])
            hints = ["One hour later means the big hand moves one step.",
                     "The minutes don't change. What hour is it now?",
                     f"Start at {h} and move the big hand one space."]
        out.append(mk(skill, i + 1, "fill_in", q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_time_one(sk, granularity, elapsed, exclude), skill, 1, a),
                      "Clocks are just a circle of hours with a minutes hand that hops by fives. You've got this!",
                      1, ["time"] + (["elapsed"] if elapsed else [])))
    return out

def gen_time_one(skill, granularity, elapsed, exclude=None):
    h = rng.randint(1, 11); m = rng.choice([0, 30])
    if elapsed:
        t = f"{h+1}:{m:02d}"
        q = f"It is {h}:{m:02d}. What time is it after 1 hour?"
        s = steps([dict(expression=None, result=t, explanation="hour up by one")])
    else:
        t = f"{h}:{m:02d}"; q = f"The clock shows {h} and the little hand is on the 6. What time is it?"
        s = steps([dict(expression=None, result=t, explanation="little hand on 6 = 30 minutes past")])
    if t == exclude: return None
    return {"question_text": q, "answer": t, "solution_steps": s}

# ---------------- fractions ----------------
def gen_fractions(skill, n, denom_choices, of_set=True, combine=False, draw_denoms=None):
    out = []
    for i in range(n):
        kind = rng.random()
        if combine and kind < 0.3:  # combine to a whole (Y3)
            d = rng.choice(denom_choices)
            k = rng.randint(1, d - 1)
            q = f"{k}/{d} + {d-k}/{d} = ?  (two fractions with the same bottom number add to what?)"
            a = "one whole"; check = {"kind": "word", "value": "one whole", "numeric_also": 1}
            s = steps([dict(expression=f"{k}/{d} + {d-k}/{d}", result="1",
                            explanation="same bottom number: add the tops; if the top equals the bottom, that's one whole")])
            check["numeric_also"] = 1
            hints = ["When the bottom numbers match, you only add the tops.",
                     "Look at the tops (the little numbers). Add just those.",
                     f"When the top equals the bottom ({d}/{d}), that's how much?"]
            story = f"A chocolate bar is split into equal pieces. You eat some, your sibling eats the rest. Together you ate how many of the bar?"
        elif of_set and kind < 0.75:  # fraction of a set
            d = rng.choice(denom_choices)
            total = d * rng.randint(2, max(2, 30 // d))
            q = rng.choice([
                f"There are {total} {plural(rng.choice(COUNTERS))}. What is 1/{d} of them?",
                f"Share {total} {plural(rng.choice(COUNTERS))} equally into {d} groups. How many in each group?",
            ])
            a = str(total // d); check = num(total // d)
            s = steps([dict(expression=f"{total} / {d}", result=total // d,
                            explanation=f"split into {d} equal groups")])
            hints = [f"1/{d} means split into {d} equal groups and take ONE group.",
                     "How many fit in each equal pile?",
                     f"Count them out in {d} equal piles."]
            story = f"Split them into equal bags, one bag for each friend. How many in each bag?"
        else:  # draw / describe (denominators constrained to the skill's scope)
            d = rng.choice(draw_denoms or denom_choices)
            q = f"Draw a shape split into {d} equal parts. Shade 1/{d}."
            a = f"shape divided into {d} equal parts with 1 shaded"; check = drawing(f"1/{d} shaded")
            s = steps([dict(expression=None, result=f"{d} equal parts, 1 shaded",
                            explanation=f"equal parts are the key — 1 out of {d}")])
            hints = ["First draw the shape — a square or circle works.",
                     f"Split it into {d} EQUAL parts.",
                     "Shade exactly one of those parts."]
            story = f"Imagine a pizza cut into {d} equal slices — you take one. Draw your slice!"
        out.append(mk(skill, i + 1,
                      "drawing" if check["kind"] == "drawing" else ("word_problem" if "of them" in q or "groups" in q else "fill_in"),
                      q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_fractions_one(sk, denom_choices, combine, exclude), skill, 1, a),
                      story, 1, ["fractions", f"denom_{d}"]))
    return out

def gen_fractions_one(skill, denom_choices, combine, exclude=None):
    d = rng.choice(denom_choices); total = d * rng.randint(2, 6)
    v = total // d
    if str(v) == exclude: return None
    return {"question_text": f"What is 1/{d} of {total}?", "answer": str(v),
            "solution_steps": steps([dict(expression=f"{total}/{d}", result=v, explanation="equal groups")])}

# ---------------- times tables & division ----------------
def gen_times(skill, n, tables):
    out = []
    for i in range(n):
        t = rng.choice(tables); x = rng.randint(2, 10)   # x=1 is trivial and would leak via the story
        ans = t * x
        kind = rng.random()
        if kind < 0.4:
            q = f"{t} × {x} = ?"
        elif kind < 0.75:
            q = f"There are {t} shelves and {x} books on each shelf. How many books in total?"
        else:
            q = f"Count by {t}s: {t}, {2*t}, … What is the {x}th one?"
        a = str(ans); check = num(ans)
        s = steps([dict(expression=f"{t} × {x}", result=ans,
                        explanation=f"{t} groups of {x}" if kind >= 0.4 else "times table")])
        hints = ["Draw little groups — one group for each shelf (or each jump).",
                 f"Put {x} dots in each group.",
                 f"Now count all the dots in your {t} groups." if t > 2 else f"Double it: what is {x} + {x}?"]
        story = f"Picture {t} little baskets at {rng.choice(PLACES)}, each with {x} {plural(rng.choice(COUNTERS))}. Can you count them all without touching?"
        out.append(mk(skill, i + 1, "word_problem" if kind >= 0.4 else "fill_in", q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_times_one(sk, tables, exclude), skill, 1, a),
                      story, 1, [f"times_{t}", "multiplication"]))
    return out

def gen_times_one(skill, tables, exclude=None):
    t = rng.choice(tables); x = rng.randint(1, 10); ans = t * x
    if ans == exclude: return None
    return {"question_text": f"{t} × {x} = ?", "answer": str(ans),
            "solution_steps": steps([dict(expression=f"{t}×{x}", result=ans, explanation="groups")])}

def gen_division(skill, n):
    out = []
    for i in range(n):
        g = rng.randint(2, 6); per = rng.randint(2, 6)
        total = g * per
        mode = rng.random()
        if mode < 0.4:
            q = f"Share {total} {plural(rng.choice(COUNTERS))} equally between {g} friends. How many does each friend get?"
            ans, expr = per, f"{total} / {g}"
        elif mode < 0.7:
            q = f"{total} ÷ {g} = ?"
            ans, expr = per, f"{total} / {g}"
        else:
            q = f"A bag has {total} lollies. If they are packed into groups of {per}, how many groups?"
            ans, expr = g, f"{total} / {per}"
        a = str(ans); check = num(ans)
        s = steps([dict(expression=expr, result=ans, explanation="equal sharing")])
        hints = ["Make equal piles. Count out one at a time until all piles match.",
                 "Keep going until every pile is the same height.",
                 f"Which times fact helps?  {g} × ? = {total}" if ans == per else f"{per} × ? = {total}"]
        story = f"Sharing time at {rng.choice(PLACES)}! Make sure everyone gets exactly the same — that's what fair means."
        out.append(mk(skill, i + 1, "word_problem" if "friends" in q else "fill_in", q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_division_one(sk, exclude), skill, 1, a),
                      story, 2, ["division", "equal_sharing"]))
    return out

def gen_division_one(skill, exclude=None):
    for _ in range(20):
        g = rng.randint(2, 5); per = rng.randint(2, 6)
        if str(per) == str(exclude): continue
        return {"question_text": f"Share {g*per} sweets equally between {g} friends. Each gets…?",
                "answer": str(per),
                "solution_steps": steps([dict(expression=f"{g*per}/{g}", result=per, explanation="equal piles")])}
    return None

# ---------------- odd / even ----------------
def gen_odd_even(skill, n):
    out = []
    for i in range(n):
        v = rng.randint(10, 999)
        even = v % 2 == 0
        a = "even" if even else "odd"
        q = rng.choice([
            f"Is {v} odd or even? How can you tell?",
            f"Sort these into odd and even: {v}. Which bucket does {v} go in?",
        ])
        s = steps([dict(expression=None, result=a,
                        explanation=(f"look at the ones digit: {v%10} — " + ("it's in 0,2,4,6,8 → even" if even else "it's in 1,3,5,7,9 → odd")))])
        hints = ["Only the ones digit (the last one) decides.",
                 "Is the last digit in the 0-2-4-6-8 family, or the 1-3-5-7-9 family?",
                 "One family can be split into two equal groups with nothing left over. Which family is yours?"]
        story = f"Think of pairing up {v} little penguins. If everyone finds a partner, all are together; if one is left out, that tells you the family."
        out.append(mk(skill, i + 1, "mcq", q, a, word(a), s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_odd_even_one(sk, exclude), skill, 1, a),
                      story, 1, ["odd_even"]))
    return out

def gen_odd_even_one(skill, exclude=None):
    v = rng.randint(11, 99)
    a = "even" if v % 2 == 0 else "odd"
    if a == exclude: return None
    return {"question_text": f"Is {v} in the even family or the odd family?", "answer": a,
            "solution_steps": steps([dict(expression=None, result=a, explanation="ones digit")])}

# ---------------- shapes ----------------
def gen_shapes(skill, n):
    bank = [
        ("mcq", "Which shape has 3 corners and 3 sides?", "triangle", word("triangle"),
         ["Count the corners on each shape in your head.", "Its name starts with a word that means 'three'."],
         "A slice of pizza is a classic one!"),
        ("mcq", "Which shape has 4 equal sides and 4 right angles?", "square", word("square"),
         ["Count the sides first — how many?", "Now check: are they all the SAME length?"],
         "A window is often one of these!"),
        ("drawing", "Draw a shape with 4 sides that are all the same length.", "square", drawing("square"),
         ["Start with 4 equal lines.", "Connect them corner to corner."],
         "Imagine a little floor tile with four equal sides."),
        ("mcq", "Which shape looks like a ball?", "sphere", word("sphere"),
         ["It has no corners and no flat faces.", "Round in every direction, like a football."],
         "A football is one of these!"),
        ("mcq", "Which shape has 6 faces, all squares?", "cube", word("cube"),
         ["A die (the dice in a board game) is the best clue.", "Count the faces you can see, then imagine the hidden ones."],
         "Think of a die from a board game — that's the shape!"),
        ("fill_in", "A rectangle has how many sides?", "4", num(4),
         ["Count them: top, bottom, left, right.", "Rectangles have pairs of equal sides."],
         "Your maths book is a rectangle!"),
        ("fill_in", "A triangle has how many corners?", "3", num(3),
         ["Point at the top, then the bottom-left, then the bottom-right.", "Count them."],
         "A play triangle on the playground!"),
        ("mcq", "Which shape has 5 corners?", "pentagon", word("pentagon"),
         ["PENT means five.", "Draw 5 points and join them."],
         "Some footy goal posts look like this!"),
        ("fill_in", "A square has how many corners?", "4", num(4),
         ["Corners are where two sides meet.", "Count them one by one, slowly."],
         "A garden bed is often one of these!"),
        ("mcq", "Which flat shape has NO corners?", "circle", word("circle"),
         ["Corners need two straight sides meeting.", "This one has none — it's all one curve."],
         "A plate is one of these!"),
        ("fill_in", "A cube has how many faces?", "6", num(6),
         ["Count the faces you can see.", "Now imagine the hidden ones — top and bottom, front and back, left and right."],
         "A die has exactly that many!"),
        ("mcq", "Which shape is like a can — two round faces and a curved side?", "cylinder", word("cylinder"),
         ["Two flat round faces.", "One smooth side joining them."],
         "A soup can is one of these!"),
        ("fill_in", "A pentagon has how many sides?", "5", num(5),
         ["PENT means five.", "Count the sides to check."],
         "Think of a shape whose name starts with PENT!"),
        ("mcq", "Which shape has 8 sides?", "octagon", word("octagon"),
         ["OCT means eight.", "Count round the shape to check."],
         "Some stop signs look like this!"),
        ("fill_in", "A pentagon has how many corners?", "5", num(5),
         ["Corners and sides come in pairs here.", "Count the corners to check."],
         "Think of a shape whose name starts with PENT!"),
        ("drawing", "Draw a shape with 3 sides and 3 corners.", "triangle", drawing("triangle"),
         ["Start with a line at the bottom.", "Join two more lines to make a peak."],
         "A slice of pizza is a classic one!"),
    ]
    out = []
    for i in range(n):
        b = rng.choice(bank)
        hints = [b[4][0], b[4][1], "Trust your counting — corners and sides never lie."]
        # similar example: a DIFFERENT shape question with a DIFFERENT answer
        sim = next((x for x in bank if x is not b and x[2] != b[2]), bank[0])
        out.append(mk(skill, i + 1, b[0], b[1], b[2], b[3],
                      steps([dict(expression=None, result=b[2], explanation="count sides and corners")]),
                      hints,
                      {"question": sim[1], "answer": sim[2],
                       "steps": [dict(step=1, expression=None, result=sim[2], explanation="count sides and corners")]},
                      b[5], 1, ["shapes"]))
    return out

# ---------------- number ordering / reading (Y1 100, Y2 1000, Y3 10000) ----------------
def gen_number_read(skill, n, cap):
    out = []
    digits = len(str(cap))
    lo = 10 ** (digits - 1)
    if cap <= 100: lo, cap_hi = 20, 99
    else: cap_hi = cap
    words = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 7: "seven",
             8: "eight", 9: "nine", 10: "ten", 11: "eleven", 12: "twelve", 13: "thirteen",
             14: "fourteen", 15: "fifteen", 16: "sixteen", 17: "seventeen", 18: "eighteen", 19: "nineteen",
             20: "twenty", 30: "thirty", 40: "forty", 50: "fifty", 60: "sixty", 70: "seventy",
             80: "eighty", 90: "ninety", 100: "one hundred", 200: "two hundred", 300: "three hundred",
             400: "four hundred", 500: "five hundred", 600: "six hundred", 700: "seven hundred",
             800: "eight hundred", 900: "nine hundred", 1000: "one thousand", 2000: "two thousand",
             3000: "three thousand", 4000: "four thousand", 5000: "five thousand", 6000: "six thousand",
             7000: "seven thousand", 8000: "eight thousand", 9000: "nine thousand"}
    used = set()
    def say(v):
        if v in words: return words[v]
        parts = []
        if v >= 1000:
            parts.append(words[v // 1000 * 1000]); v %= 1000
        if v >= 100:
            parts.append(words[v // 100 * 100]); v %= 100
        if v in words:
            parts.append(words[v])
        elif v:
            parts.append(words[v // 10 * 10]); parts.append(words[v % 10])
        return " ".join(parts)
    for i in range(n):
        kind = rng.random()
        if kind < 0.4:  # write in words
            for _ in range(30):
                v = rng.randint(lo, cap_hi)
                if ("words", v) not in used: break
            used.add(("words", v))
            a = say(v)
            q = f"Write the number {v:,} in words."
            check = word(a)
            s = steps([dict(expression=None, result=a, explanation="name each place: hundreds (thousands) first, then tens, then ones")])
            hints = ["Say the biggest place value first.",
                     "Then the tens, then the ones.",
                     f"Try saying {v} out loud slowly — that's the answer!"]
        elif kind < 0.7:  # order / compare
            for _ in range(30):
                v1 = rng.randint(lo, cap_hi); v2 = rng.randint(lo, cap_hi)
                if v1 != v2 and ("order", frozenset((v1, v2))) not in used: break
            if v1 == v2: v2 = cap_hi - 3 if v1 != cap_hi - 3 else cap_hi - 7
            used.add(("order", frozenset((v1, v2))))
            big, small = max(v1, v2), min(v1, v2)
            # the answer is ONE number (the bigger), so the question must ask for one number.
            # The old "Put A and B in order, smallest first." variant kept answer = bigger only;
            # its 20 rows were reworded to this form on 2026-10-13 in every data file.
            _ = rng.choice([0, 1])            # keep the random stream identical to the original build
            q = f"Which number is bigger: {v1:,} or {v2:,}?"
            a = f"{big:,}"
            check = num(big)
            s = steps([dict(expression=f"{big} > {small}", result=f"{big:,}",
                            explanation="same number of digits? compare left to right")])
            hints = ["Same number of digits? Compare from the LEFT.",
                     "Which has the bigger digit first?",
                     "The one with the bigger first digit is the bigger number."]
        elif kind < 0.9 and cap >= 1000:  # place value of a digit
            for _ in range(30):
                v = rng.randint(10 ** (digits - 1), min(10 ** digits - 1, cap))
                if ("pv", v) not in used: break
            used.add(("pv", v))
            while True:
                pos = rng.randint(0, 3)
                digit = (v // 10 ** pos) % 10
                if digit != 0: break
            val = digit * 10 ** pos
            place = ["ones", "tens", "hundreds", "thousands"][pos]
            q = f"What is the value of the {digit} in {v:,}?"
            a = f"{val:,}"; check = num(val)
            s = steps([dict(expression=f"{digit} × {10**pos:,}", result=val, explanation=f"the {place} place is worth {10**pos:,}")])
            hints = [f"That digit is in the {place} place.",
                     f"Each {place} place is worth {10**pos:,}.",
                     f"So {digit} of them is… work it out!"]
        else:  # round (value must not already be a multiple, else the question is a trick)
            unit = rng.choice([10, 100])
            for _ in range(30):
                v = rng.randint(lo, cap_hi)
                if v % unit and ("round", (v, unit)) not in used: break
            used.add(("round", (v, unit)))
            # nearest, halves round UP (Australian school convention). This used to be
            # (v // unit) * unit - always DOWN - which made 17 bank answers wrong
            # (e.g. 97 -> 90); fixed 2026-10-13 and patched in every derived data file.
            r = (v + unit // 2) // unit * unit
            q = f"Round {v:,} to the nearest {unit}."
            a = f"{r:,}"; check = num(r)
            s = steps([dict(expression=f"{v} → {r}", result=r,
                            explanation="5 or more in the deciding digit rounds up, otherwise stay")])
            hints = ["Look at the digit just to the right of the rounding place — it decides up or stay.",
                     "5 or more rounds UP, less than 5 stays.",
                     "Slide to the closest round number on the number line."]
        story = f"Numbers are like addresses — the position of each digit tells you how much it's worth."
        out.append(mk(skill, i + 1, "fill_in" if "Round" in q or "value" in q else "mcq", q, a, check, s, hints,
                      _retry_similar(lambda sk, sq, exclude=None: gen_number_read_one(sk, cap, exclude), skill, 1, a),
                      story, 2 if cap >= 1000 else 1, ["number_sense", f"to_{cap}"]))
    return out

def gen_number_read_one(skill, cap, exclude=None):
    lo = 20 if cap <= 100 else max(100, 10 ** (len(str(cap)) - 1))
    cap_hi = 99 if cap <= 100 else cap
    v1 = rng.randint(lo, cap_hi); v2 = rng.randint(lo, cap_hi)
    if v1 == v2: v2 = cap_hi - 3 if v1 != cap_hi - 3 else cap_hi - 7
    big = max(v1, v2)
    if str(big) == exclude: return None
    return {"question_text": f"Which is bigger: {v1} or {v2}?", "answer": f"{big:,}",
            "solution_steps": steps([dict(expression=f"{big}>{v1 if big!=v2 else v2}", result=big, explanation="compare left to right")])}

# ---------------- missing values (Y3) ----------------
def gen_missing(skill, n):
    out = []
    for i in range(n):
        op = rng.choice("+-")
        a = rng.randint(10, 980); b = rng.randint(5, 400)
        if op == "+":
            c = a + b; hidden = rng.choice([0, 1])
            if hidden == 0:
                q = f"? + {b} = {c}"; missing = a
                s = steps([dict(expression=f"{c} - {b}", result=a,
                                explanation="the missing part is what's left when you take the known part away")])
            else:
                q = f"{a} + ? = {c}"; missing = b
                s = steps([dict(expression=f"{c} - {b if hidden==1 else a}", result=missing,
                                explanation="subtract the part you know from the total")])
        else:
            c = max(0, a - b); q = f"{a} - ? = {c}"; missing = a - c
            s = steps([dict(expression=f"{a} - {c}", result=missing, explanation="take the answer away from the start to find the jump")])
        a_txt = str(missing); check = num(missing)
        out.append(mk(skill, i + 1, "fill_in", q, a_txt, check, s,
                      ["The total (or the start) is the big number you can see.",
                       "Take the part you know AWAY from the total.",
                       "Check your answer by putting it back into the number sentence."]
                      , _retry_similar(lambda sk, sq, exclude=None: gen_missing_one(sk, exclude), skill, 1, a_txt),
                      "Missing-value questions are treasure hunts: the total is hiding a piece. Dig it out!",
                      2, ["missing_value", "inverse_ops"]))
    return out

def gen_missing_one(skill, exclude=None):
    a = rng.randint(20, 500); b = rng.randint(5, 100); c = a + b
    m = rng.choice([a, b])
    if str(m) == exclude: return None
    q = f"? + {b} = {c}" if m == a else f"{a} + ? = {c}"
    return {"question_text": q, "answer": str(m),
            "solution_steps": steps([dict(expression=f"{c} - {a if m==b else b}", result=m, explanation="subtract known from total")])}

print("generators defined")


generators defined


## 4. Assemble the bank


In [5]:
GENERATORS = {
    "y1-number-100":      lambda s, n: gen_number_read(s, n, 100),
    "y1-place-value":     lambda s, n: gen_place_value(s, n, 2),
    "y1-counting":        lambda s, n: gen_counting(s, n, 50, rng.choice([2, 5, 10]), "skip_count"),
    "y1-add-sub-20":      lambda s, n: gen_add_sub(s, n, 1, 15, 1, 15, False),
    "y1-add-sub-100":     lambda s, n: gen_add_sub(s, n, 11, 79, 11, 69, False),
    "y1-fraction-half":   lambda s, n: gen_fractions(s, n, [2], of_set=True, combine=False, draw_denoms=[2]),
    "y1-time":            lambda s, n: gen_time(s, n, granularity=30, elapsed=False),
    "y1-money":           lambda s, n: gen_money(s, n, 2000, with_change=False),
    "y1-shapes":          lambda s, n: gen_shapes(s, n),
    "y2-number-1000":     lambda s, n: gen_number_read(s, n, 999),
    "y2-place-value":     lambda s, n: gen_place_value(s, n, 3),
    "y2-add-sub":         lambda s, n: gen_add_sub(s, n, 15, 499, 5, 399, True),
    "y2-times-2":         lambda s, n: gen_times(s, n, [2]),
    "y2-fractions":       lambda s, n: gen_fractions(s, n, [2, 3, 4], of_set=True, combine=False),
    "y2-time":            lambda s, n: gen_time(s, n, granularity=15, elapsed=False),
    "y2-money":           lambda s, n: gen_money(s, n, 10000, with_change=False),
    "y2-patterns":        lambda s, n: gen_patterns(s, n),
    "y3-number-10000":    lambda s, n: gen_number_read(s, n, 9999),
    "y3-odd-even":        lambda s, n: gen_odd_even(s, n),
    "y3-add-sub":         lambda s, n: gen_add_sub(s, n, 120, 8999, 120, 6999, True),
    "y3-missing-value":   lambda s, n: gen_missing(s, n),
    "y3-times":           lambda s, n: gen_times(s, n, [3, 4, 5, 10]),
    "y3-division":        lambda s, n: gen_division(s, n),
    "y3-fractions-set":   lambda s, n: gen_fractions(s, n, [2, 3, 4, 5, 10], of_set=True, combine=True),
    "y3-time":            lambda s, n: gen_time(s, n, granularity=5, elapsed=True),
    "y3-money":           lambda s, n: gen_money(s, n, 5000, with_change=True),
}

GLOBAL_SEEN_TEXTS = set()  # cross-skill text uniqueness (protects M3's split isolation)

def assemble_skill(skill, gen, target):
    """Draw problems until the question text is unique within AND across skills."""
    seen = set()
    out = []
    tries = 0
    while len(out) < target and tries < target * 60:
        tries += 1
        p = gen(skill, 1)[0]
        key = re.sub(r"\s+", " ", p["question_text"].strip().lower())
        if key in seen or key in GLOBAL_SEEN_TEXTS: continue
        seen.add(key)
        GLOBAL_SEEN_TEXTS.add(key)
        p["id"] = f"geeta-{skill['id']}-{len(out)+1:04d}"
        out.append(p)
    if len(out) < target:
        raise RuntimeError(f"could not generate {target} unique problems for {skill['id']}")
    return out

BANK_PROBLEMS = []
for skill in SKILLS:
    BANK_PROBLEMS.extend(assemble_skill(skill, GENERATORS[skill["id"]], skill["target"]))
print(f"bank size: {len(BANK_PROBLEMS)} problems")


bank size: 1090 problems


In [6]:
# ---------------- build-time QA ----------------
# Global text deduplication (cross-skill): the same question text generated by two
# skills (e.g. 'What number is 7 tens and 7 ones?' in y1-place-value AND
# y2-place-value, or the same shading drawing in two fraction skills) would break
# M3's text-level split isolation. Keep the first occurrence in skill order.
_seen_global = set()
_kept = []
for p in BANK_PROBLEMS:
    k = re.sub(r'\s+', ' ', p['question_text'].strip().lower())
    if k in _seen_global:
        continue
    _seen_global.add(k)
    _kept.append(p)
_dropped = len(BANK_PROBLEMS) - len(_kept)
BANK_PROBLEMS[:] = _kept
print(f'global text dedup: dropped {_dropped} cross-skill duplicate(s) -> {len(BANK_PROBLEMS)} problems')

errors = []
seen_q = {}
counts = {}
for p in BANK_PROBLEMS:
    counts[p["skill_id"]] = counts.get(p["skill_id"], 0) + 1
    key = p["skill_id"] + "|" + p["question_text"].strip().lower()
    if key in seen_q:
        errors.append(f"DUPLICATE question: {key[:70]}")
    seen_q[key] = p["id"]
    # hint leak check
    leaked = hints_leak(p["hint_ladder"], p["answer_check"])
    if leaked:
        errors.append(f"LEAK {p['id']}: hints contain {leaked} (answer={p['answer']})")
    # story is shown to the child, so it must not leak either
    story_leak = hints_leak([p["story_reframe"]], p["answer_check"])
    if story_leak:
        errors.append(f"STORY-LEAK {p['id']}: story contains {story_leak} (answer={p['answer']})")
    # similar example must differ in final answer
    if str(p["similar_worked"]["answer"]).strip().lower() == str(p["answer"]).strip().lower():
        errors.append(f"SIMILAR-EQ-ANSWER {p['id']}")
    # numeric scope sanity for number-kind answers
    ck = p["answer_check"]
    if ck["kind"] == "number":
        if ck["value"] < 0 or ck["value"] > 20000:
            errors.append(f"OUT-OF-SCOPE {p['id']} value={ck['value']}")

# per-skill minimums
for skill in SKILLS:
    if counts.get(skill["id"], 0) < max(10, skill["target"] - 5):
        errors.append(f"LOW-COUNT {skill['id']}: {counts.get(skill['id'])}/{skill['target']}")

print(f"QA: {len(errors)} issue(s)")
for e in errors[:20]:
    print("  !", e)
assert not errors, "bank QA failed"
print("bank QA passed ✔")


global text dedup: dropped 0 cross-skill duplicate(s) -> 1090 problems
QA: 0 issue(s)
bank QA passed ✔


In [7]:
# ---------------- write outputs ----------------
with open(BANK / "acara_y1-3.jsonl", "w") as f:
    for p in BANK_PROBLEMS:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

with open(BANK / "skill_map.json", "w") as f:
    json.dump({
        "source": "ACARA Mathematics Curriculum F–10 (v9.0), Years 1–3, in-scope strands",
        "retrieved": "2026-08-18",
        "code_verification": {
            "verified": "confirmed against QCAA-published ACARA v9.0 alignment documents",
            "approximate": "follows v9.0 code convention; re-verify at australiancurriculum.edu.au before publication",
        },
        "skills": SKILLS,
    }, f, indent=2, ensure_ascii=False)

per_skill = sorted(counts.items(), key=lambda kv: kv[0])
md = ["# ACARA Years 1–3 problem bank — summary", "",
      f"- generated: {datetime.date.today().isoformat()} (seed {SEED})",
      f"- total problems: **{len(BANK_PROBLEMS)}**", "",
      "| skill | year | ACARA code | status | problems |", "|---|---|---|---|---|"]
for s in SKILLS:
    md.append(f"| {s['name']} | Y{s['year']} | {s['code']} | {s['code_status']} | {counts[s['id']]} |")
md += ["", "QA: hints leak-checked; similar examples answer-differed; numeric scope checked; no duplicate questions.",
       "Out of scope (not generated): percentages, negatives, decimals money, Year 4+ content."]
with open(BANK / "bank_summary.md", "w") as f:
    f.write("\n".join(md) + "\n")

print(f"wrote:\n  {BANK/'acara_y1-3.jsonl'}\n  {BANK/'skill_map.json'}\n  {BANK/'bank_summary.md'}")

wrote:
  /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/bank/acara_y1-3.jsonl
  /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/bank/skill_map.json
  /Users/kiruthikkumaar/Downloads/Bionic/GEETA - 4B/data/bank/bank_summary.md
